In [1]:
import os
import numpy as np
import pandas as pd
%pip install pulp
import pulp
import time

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Loading processed data

current_dir = os.getcwd()
data_dir = os.path.join(os.path.dirname(current_dir), 'data', 'processed')

csv_file_path = os.path.join(data_dir, "cleaned_delivery_orders.csv")
npz_file_path = os.path.join(data_dir, "vrp_mathematical_matrices.npz")

df = pd.read_csv(csv_file_path)
print("DataFrame loaded successfully.")

matrices = np.load(npz_file_path, allow_pickle=True)
print("Matrices loaded successfully.And available matrices are:", matrices.files)

cargo_weights=matrices['cargo_weights']
order_values=matrices['order_values']
service_times=matrices['service_times']
window_starts=matrices['window_starts']
window_ends=matrices['window_ends']
distance_matrix_km=matrices['distance_matrix_km']
time_cost_matrix=matrices['time_cost_matrix']
vehicle_capacities = matrices['vehicle_capacities']
vehicle_shifts_min = matrices['vehicle_shifts_min']

print('All matrices loaded successfully.')

DataFrame loaded successfully.
Matrices loaded successfully.And available matrices are: ['cargo_weights', 'order_values', 'service_times', 'window_starts', 'window_ends', 'distance_matrix_km', 'time_cost_matrix', 'vehicle_capacities', 'vehicle_shifts_min']
All matrices loaded successfully.


In [3]:
# scalability slicer
N = 20
slice_limit = N + 1

# Slice 1D arrays
sliced_cargo_weights = cargo_weights[:slice_limit]
sliced_order_values = order_values[:slice_limit]
sliced_service_times = service_times[:slice_limit]
sliced_window_starts = window_starts[:slice_limit]
sliced_window_ends = window_ends[:slice_limit]

# Slice 2D arrays
sliced_distance_matrix_km = distance_matrix_km[:slice_limit, :slice_limit]
sliced_time_cost_matrix = time_cost_matrix[:slice_limit, :slice_limit]

# Slice the DataFrame
sliced_df = df.iloc[:slice_limit].copy()

print("Sliced DataFrame and matrices successfully.")
print("Sliced DataFrame shape:", sliced_df.shape)
print("Sliced cargo_weights shape:", sliced_cargo_weights.shape)
print("Sliced order_values shape:", sliced_order_values.shape)
print("Sliced service_times shape:", sliced_service_times.shape)
print("Sliced window_starts shape:", sliced_window_starts.shape)
print("Sliced window_ends shape:", sliced_window_ends.shape)
print("Sliced distance_matrix_km shape:", sliced_distance_matrix_km.shape)
print("Sliced time_cost_matrix shape:", sliced_time_cost_matrix.shape)


Sliced DataFrame and matrices successfully.
Sliced DataFrame shape: (21, 12)
Sliced cargo_weights shape: (21,)
Sliced order_values shape: (21,)
Sliced service_times shape: (21,)
Sliced window_starts shape: (21,)
Sliced window_ends shape: (21,)
Sliced distance_matrix_km shape: (21, 21)
Sliced time_cost_matrix shape: (21, 21)


In [4]:
# Initialize the Engine
model = pulp.LpProblem("VRP_Profit_Maximization", pulp.LpMaximize)
print("Model initialized successfully.")

# Define dimensions
num_vehicles = 2
V = range(slice_limit)  # Number of orders (including depot)
K = range(num_vehicles)  # Number of vehicles

x = pulp.LpVariable.dicts("route", (V, V, K), cat=pulp.LpBinary)
print("Decision variables defined successfully.")

# Create the routing variable
y = pulp.LpVariable.dicts("select", (V, K), cat=pulp.LpBinary)
print("Routing variables defined successfully.")


# Create time variable
t = pulp.LpVariable.dicts("time", (V, K), lowBound=0, cat=pulp.LpContinuous)
print("Time variables defined successfully.")

print("All variables defined successfully.")
print("Number of orders (including depot):", slice_limit)
print("Number of vehicles:", num_vehicles)


Model initialized successfully.
Decision variables defined successfully.
Routing variables defined successfully.
Time variables defined successfully.
All variables defined successfully.
Number of orders (including depot): 21
Number of vehicles: 2


In [ ]:
# Force all arrays to be pure floats so PuLP can perform algebra on them
sliced_cargo_weights = sliced_cargo_weights.astype(float)
sliced_order_values = sliced_order_values.astype(float)
sliced_service_times = sliced_service_times.astype(float)
sliced_window_starts = sliced_window_starts.astype(float)
sliced_window_ends = sliced_window_ends.astype(float)
sliced_distance_matrix_km = sliced_distance_matrix_km.astype(float)
sliced_time_cost_matrix = sliced_time_cost_matrix.astype(float)
sliced_priorities = df['Priority_Level'].to_numpy()[:slice_limit].astype(int)


num_vehicles = vehicle_capacities = matrices['vehicle_capacities']
vehicle_shifts_min = matrices['vehicle_shifts_min']
num_vehicles = len(vehicle_capacities)
V = range(slice_limit) 
K = range(num_vehicles)


# mocking a basic Vehicle_Fleet table for our 2 test vehicles
vehicle_capacities = [4000, 4000]  # Example capacities for 2 vehicles
vehicle_shifts_min = [720,720]  # Example shifts in minutes for 2 vehicles

#  MODEL & VARIABLE INITIALIZATION
model = pulp.LpProblem("VRP_Profit_Maximization", pulp.LpMaximize)

x = pulp.LpVariable.dicts("route", (V, V, K), cat=pulp.LpBinary)
y = pulp.LpVariable.dicts("select", (V, K), cat=pulp.LpBinary)
t = pulp.LpVariable.dicts("time", (V, K), lowBound=0, cat=pulp.LpContinuous)


# Define the objective function
penalty_value = 10000 # Penalty for not serving an order

objective = (
    pulp.lpSum(y[i][k] * sliced_order_values[i] for i in V for k in K) -
    pulp.lpSum(x[i][j][k] * sliced_time_cost_matrix[i][j] for i in V for j in V for k in K)
    
    - pulp.lpSum(
        penalty_value * (1 - pulp.lpSum(y[i][k] for k in K)) for i in V if  sliced_priorities[i] == 5 and i != 0
    )
)
model += objective, "Maximize_Profit"
print("Objective function defined successfully.")


# Mathematical Constraints
M = 1000000

for k in K:
    model += pulp.lpSum(y[i][k] * sliced_cargo_weights[i] for i in V) <= vehicle_capacities[k], f"Cap_{k}"
    
    model += (
        pulp.lpSum(y[i][k] * sliced_service_times[i] for i in V) +
        pulp.lpSum(x[i][j][k] * sliced_time_cost_matrix[i][j] for i in V for j in V if i != j)
    ) <= vehicle_shifts_min[k], f"Shift_{k}"
    
    
    for i in V:
        model += pulp.lpSum(x[i][j][k] for j in V if j != i) == y[i][k], f"Leave_order_{i}_vehicle_{k}"
        model += pulp.lpSum(x[j][i][k] for j in V if j != i) == y[i][k], f"Enter_order_{i}_vehicle_{k}"
        
    model += pulp.lpSum(y[i][k] * sliced_cargo_weights[i] for i in V) <= vehicle_capacities[k], f"Capacity_Veh_{k}"
    
    model += (
        pulp.lpSum(y[i][k] * sliced_service_times[i] for i in V) +
        pulp.lpSum(x[i][j][k] * sliced_time_cost_matrix[i][j] for i in V for j in V if i != j)
    ) <= vehicle_shifts_min[k], f"Shift_Time_Veh_{k}"
    
# Add time window constraints
for k in K:
    for i in V:
        model += t[i][k] >= sliced_window_starts[i] * y[i][k], f"Time_Window_Start_Order_{i}_Vehicle_{k}"
        model += t[i][k] <= sliced_window_ends[i] * y[i][k] + M * (1 - y[i][k]), f"Time_Window_End_Order_{i}_Vehicle_{k}"
        
        # link routing and time variables
        for j in V:
            if i != 0 and j != 0 and i != j:
                model += t[j][k] >= t[i][k] + sliced_service_times[i] + sliced_time_cost_matrix[i][j] - M * (1 - x[i][j][k]), f"Time_Link_Order_{i}_to_{j}_Vehicle_{k}"
                
#for at least one vehicle leave the depot
model += pulp.lpSum(y[0][k] for k in K) >= 1, "At_least_one_vehicle_leaves_depot"
           
print("All constraints added successfully.")

Objective function defined successfully.


IndexError: list index out of range

In [ ]:
# Start the Benchmarking Timer
start_time = time.time()

print("Starting to solve the model...")

model.solve(pulp.PULP_CBC_CMD(msg=False)) 

# Stop the timer and calculate the elapsed time
end_time = time.time()
elapsed_time = end_time - start_time

# log and calculate the KPIs
status = pulp.LpStatus[model.status]
print('Execution Results')
print('Status:', status)
print('Execution Time (seconds):', elapsed_time)

if status == 'Optimal':
    # calculate time revenue
    calculated_revenue = sum(
        sliced_order_values[i] for i in V for k in K if pulp.value(y[i][k]) == 1
    )
    print('Calculated Revenue LKR:', calculated_revenue)
    
    # Determine routing sucess
    route_nodes = []
    deferred_nodes = []

    for i in V:
        if i == 0:  # Exclude depot
            continue
        is_visited = sum(pulp.value(y[i][k]) for k in K)
    
        if is_visited > 0.5:
            route_nodes.append(i)
        else:
            deferred_nodes.append(i)
    print(f'Sucessfully Routed Orders: {route_nodes}')
    print(f'Deferred Orders: {deferred_nodes}')
else:
    print('No optimal solution found. Please check the model and constraints.')


In [ ]:
# Write the model to a file to inspect the equations
model.writeLP("vrp_debug.lp")
print("Model written to vrp_debug.lp. Open this file to see the constraints.")

# Print the number of constraints to ensure they were actually added
print(f"Total constraints in model: {len(model.constraints)}")

In [ ]:
results_log = []

for N in [10, 15, 20]:
    slice_limit = N + 1
    V = range(slice_limit)
    K = range(2) # Vehicles
    
    # [PASTE YOUR MODEL INITIALIZATION & CONSTRAINTS HERE]
    # (Just ensure 'sliced_' variables use the N from the loop)
    
    model.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=60))
    
    # Record results
    routed_count = sum(1 for i in V if i != 0 and sum(pulp.value(y[i][k]) for k in K) > 0.5)
    results_log.append({'N': N, 'Status': pulp.LpStatus[model.status], 'Routed_Count': routed_count})

# Display summary
print(pd.DataFrame(results_log))